# Data Generation

## Imports

In [26]:
import os
import numpy as np
import pandas as pd
from scipy.io import loadmat

## Directories

In [27]:
base_dir = os.getcwd()
data_dir = os.path.join(base_dir, 'data')

processed_data_dir = os.path.join(base_dir, 'processed_data')
os.makedirs(processed_data_dir, exist_ok=True)

## Data generation settings

In [38]:
T_limit = 1000
train_val_test_split = [0.7, 0.15, 0.15]
window_length, stride = 30, 1

## Load Data

In [29]:
ampds_folderpath = os.path.join(data_dir, 'ampds')
ampds_filepath = os.path.join(ampds_folderpath, 'ampds2.mat')

data = loadmat(ampds_filepath)

x, y = data['input'], data['output']
input_labels, output_labels = data['labelInp'], data['labelOut']

T = x.shape[0]
T_limit = min(T, T_limit) if T_limit is not None else T
x, y = x[:T_limit], y[:T_limit]

x_df = pd.DataFrame(x, columns=input_labels)
y_P_df = pd.DataFrame(y[:,:,0], columns=output_labels)

print("Input shape:", x_df.shape)
print("Output shape:", y_df.shape)
print("Input missing values:")
display(x_df.isna().sum())
print("Output missing values:")
display(y_df.isna().sum())

display(x_df.head())
display(y_df.head())

Input shape: (1000, 6)
Output shape: (1000, 22)
Input missing values:


time     0
id       0
P-agg    0
I-agg    0
Q-agg    0
S-agg    0
dtype: int64

Output missing values:


time    0
id      0
RSE     0
GRE     0
B1E     0
BME     0
CWE     0
DWE     0
EQE     0
FRE     0
HPE     0
OFE     0
UTE     0
WOE     0
B2E     0
CDE     0
DNE     0
EBE     0
FGE     0
HTE     0
OUE     0
TVE     0
dtype: int64

,time,id,P-agg,I-agg,Q-agg,S-agg
0,60.0,1.0,918.0,8.3,141.0,1037.0
1,120.0,1.0,913.0,8.2,141.0,1028.0
2,180.0,1.0,872.0,8.0,139.0,993.0
3,240.0,1.0,872.0,8.0,141.0,990.0
4,300.0,1.0,772.0,7.8,140.0,988.0


,time,id,RSE,GRE,B1E,BME,CWE,DWE,EQE,FRE,...,UTE,WOE,B2E,CDE,DNE,EBE,FGE,HTE,OUE,TVE
0,60.0,1.0,43.0,2.0,0.0,352.0,0.0,0.0,41.0,106.0,...,9.0,0.0,58.0,0.0,30.0,0.0,0.0,4.0,0.0,37.0
1,120.0,1.0,43.0,0.0,0.0,354.0,0.0,0.0,35.0,107.0,...,9.0,0.0,58.0,0.0,30.0,0.0,0.0,4.0,0.0,37.0
2,180.0,1.0,43.0,2.0,0.0,354.0,0.0,0.0,41.0,107.0,...,9.0,0.0,8.0,0.0,30.0,0.0,0.0,4.0,0.0,37.0
3,240.0,1.0,43.0,2.0,0.0,354.0,0.0,0.0,40.0,106.0,...,9.0,0.0,7.0,0.0,30.0,0.0,0.0,4.0,0.0,37.0
4,300.0,1.0,41.0,2.0,0.0,352.0,0.0,0.0,40.0,106.0,...,9.0,0.0,8.0,0.0,31.0,0.0,0.0,4.0,0.0,37.0


## Normalize data

Min–max normalization rescales each feature using its own minimum and maximum:

$
x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$

This maps values to roughly 0–1. The `1e-8` term prevents division by zero when a feature is constant.

It’s often used before machine-learning training so features with larger numeric ranges don’t dominate, and to help some models train more reliably. Calculate min/max on the training data, then reuse those same values for validation and test data.

In [30]:
def normalize_data(x):
    x_min = x.min(axis=0, keepdims=True)
    x_max = x.max(axis=0, keepdims=True)
    return x_min, x_max, (x - x_min) / (x_max - x_min + 1e-8)

In [31]:
x_min, x_max, x_norm = normalize_data(x)
y_min, y_max, y_norm = normalize_data(y)

scaling_factors = {
    'x_min': x_min,
    'x_max': x_max,
    'y_min': y_min,
    'y_max': y_max}

## Create NILM Dataset

### Precompute indices

We will train a supervised sequence-to-point model, using an overlapping input window to predict the target at its center. We’ll split the time series chronologically into training, validation, and test sets before generating windows, preventing overlap between sets and preserving temporal order. Storing every window at once would require too much memory, so we’ll generate windows one batch at a time. To do this efficiently, we’ll precompute each window’s indices rather than storing the windows themselves.

In [39]:
def precompute_indices(num_timesteps, window_length, stride, train_val_test_split, seed=42):
    
    num_windows = (num_timesteps - window_length + 1) // stride + 1
    inp_idx = np.arange(0, num_windows * stride, stride)
    center_offset = window_length // 2
    out_idx = inp_idx + center_offset
    
    # Avoid overflow at edges
    valid_mask = out_idx < num_timesteps
    inp_idx = inp_idx[valid_mask]
    out_idx = out_idx[valid_mask]
    
    # Shuffle indices
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(inp_idx))
    inp_idx = inp_idx[perm]
    out_idx = out_idx[perm]
    
    # Split
    n = len(inp_idx)
    n_train = int(train_val_test_split[0] * n)
    n_val = int(train_val_test_split[1] * n)
    train_inp = inp_idx[:n_train]
    train_out = out_idx[:n_train]
    val_inp = inp_idx[n_train:n_train + n_val]
    val_out = out_idx[n_train:n_train + n_val]
    test_inp = inp_idx[n_train + n_val:]
    test_out = out_idx[n_train + n_val:]
    
    return num_windows, {
        'train': (train_inp, train_out),
        'val': (val_inp, val_out),
        'test': (test_inp, test_out)}

idx_dict = precompute_indices(T, window_length, stride, train_val_test_split, seed=42)

This code builds and shuffles index pairs for the windowed time-series examples:
- `num_windows` estimates how many windows fit. `inp_idx` creates their starting indices, spaced by `stride`.
- `out_idx` points to the midpoint of each window, where the target is taken.
- `valid_mask` removes targets beyond the data's end.
- `perm` randomizes the input indices.
- Shuffle `out_idx` with the same permutation so each input stays paired with its target.

### Make PQ image

In [33]:
def make_image(x_win):
    
    x_win = tf.convert_to_tensor(x_win, dtype=tf.float32)
    p, q = x_win[:,0], x_win[:,1]
    
    # Build PQ Signature
    p_col, q_row = tf.reshape(p, (-1,1)), tf.reshape(q, (1, -1))
    S_xy = tf.sqrt(p_col**2 + q_row**2) # Compute pairwise S_xy, producing (W,W) matrix.
    top = tf.concat([tf.zeros((1,1)), q_row], axis=1) # Build first row (q only)
    left = tf.concat([p_col, S_xy], axis=1) # Build remaining rows. First column is p only.
    S = tf.concat([top, left], axis=0) # Combine into PQ image.
    S = tf.expand_dims(S, -1) # Add Channel Dimension.

    # Build FFT
    S_fft = tf.signal.fft2d(tf.cast(S[:,:,0], tf.complex64)) # Compute 2D Fourier transform of PQ image.
    S_fft = tf.abs(S_fft) # Keep magnitude only.
    S_fft = tf.expand_dims(S_fft, -1) # Add Channel Dimension

    return (S, S_fft)

### Generate batches

`make_image` converts a batch of input windows into two representations:
- It treats columns 0 and 1 as active power (p) and reactive power (q).
- It builds a pairwise PQ signature matrix, where each cell combines a `p` and `q` value using $\sqrt{p^2+q^2}$, with the original values added along the top and left edges.
- It computes the 2D Fourier Transform of that matrix and keeps only its magnitude.

It returns the PQ signature and its FFT magnitude, each with a channel dimension for model input.

In [23]:
def generate_sample(x_data, y_data, i_inp, i_out, window_length):

    x_win = x_data[i_inp : i_inp + window_length] # (W,2)
    if x_win.shape[0] != window_length: return None
    y_target = y_data[i_out]
    S, S_fft = make_image(x_win)
    return (S, S_fft), y_target

`generate_sample` creates one training example. 
- Extracts `window_length` rows from `x_data`, starting at `i_inp`, returns `None` if the window is incomplete.
- Selects the corresponding target row from `y_data` at `i_out`.
- Converts the input window into a PQ signature image and its FFT magnitude using `make_image`.
- Returns both images as inputs, paired with the target.

In [34]:
def generate_batch(x_data, y_data, inp_idx, out_idx, window_length):

    S_list = []
    S_fft_list = []
    y_list = []

    for i_inp, i_out in zip(inp_idx, out_idx):
        result = generate_sample(x_data, y_data, i_inp, i_out, window_length)
        if result is None: continue
        (S, S_fft), y = result
        S_list.append(S.numpy()) # (31, 31, 1)
        S_fft_list.append(S_fft[..., np.newaxis])
        y_list.append(y)

    return (np.array(S_list, dtype=np.float32), np.array(S_fft_list, dtype=np.float32)), np.array(y_list, dtype=np.float32)

`generate_batch` builds a batch of model examples from paired input and output indices:
- Iterates through each (`i_inp`, `i_out`) pair and calls `generate_sample`.
- Skips incomplete windows (`None` results).
- Collects each sample's PQ image, FFT image, and target.
- Converts the lists into `float32` NumPy arrays and returns `((PQ_images, FFT_images), targets)`.

So the returned structure is ready to feed into a model with two inputs and corresponding targets.

### Save dataset

In [42]:
dataset_path = os.path.join(data_dir, 'ampds_dataset.npz')

(train_inp, train_out), (val_inp, val_out), (test_inp, test_out) = idx_dict

np.savez_compressed(
    dataset_path,
    x_norm=x_norm,
    y_norm=y_norm,
    **scaling_factors,
    train_inp=train_inp, train_out=train_out,
    val_inp=val_inp, val_out=val_out,
    test_inp=test_inp, test_out=test_out)

ValueError: not enough values to unpack (expected 3, got 2)